In [1]:

# 1. Imports
from getpass import getpass
from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings
)
from langchain_text_splitters import CharacterTextSplitter
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 2. Setup Gemini API Key
GOOGLE_API_KEY = getpass("Enter Gemini API Key: ")

# 3. Setup LLM and Embeddings
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0,
    google_api_key=GOOGLE_API_KEY
)

embedding = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=GOOGLE_API_KEY
)

# 4. Load your document
with open("sample.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

# 5. Split into chunks
splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=300,
    chunk_overlap=50
)

chunks = splitter.split_text(raw_text)
documents = [Document(page_content=chunk) for chunk in chunks]

# 6. Compress documents using LLMChainExtractor
compressor = LLMChainExtractor.from_llm(llm)

compressed_docs = compressor.compress_documents(
    documents,
    query="Summarize the key points"
)

print("Compressed Summary:")
for doc in compressed_docs:
    print("-", doc.page_content)

# 7. Ask a question about compressed content
qa_prompt = PromptTemplate.from_template(
    "Given the context below, answer the question:\n\n"
    "Context:\n{context}\n\n"
    "Question:\n{question}"
)

qa_chain = qa_prompt | llm | StrOutputParser()

response = qa_chain.invoke({
    "context": "\n".join(
        doc.page_content for doc in compressed_docs
    ),
    "question": "What is the main idea of the document?"
})

print("\nAnswer to your question:")
print(response)


Enter Gemini API Key: ··········
Compressed Summary:
- LangChain is a framework for building applications with LLMs.LangChain was created by Harrison Chase.LangChain supports RAG, agents, memory, tools, and more.It’s commonly used in chatbots, document Q&A, and AI workflow

Answer to your question:
The main idea of the document is to provide an overview of **LangChain**, defining it as a framework created by Harrison Chase for building LLM applications, and outlining its key features (such as RAG, agents, memory, and tools) and common use cases (like chatbots, document Q&A, and AI workflows).


In [4]:
%pip install -q filetype

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-google-genai 4.4.3 requires google-genai>=2.20.0, but you have google-genai 2.12.1 which is incompatible.
langchain-google-genai 4.4.3 requires langchain-core<2.0.0,>=1.7.5, but you have langchain-core 1.7.4 which is incompatible.


In [2]:
%pip install -q --no-deps langchain-google-genai==4.4.3

In [2]:

%pip install -U langchain-text-splitters


In [4]:

%pip install -U langchain-classic


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 6.1 MB/s  0:00:00
